In [1]:
pip install ugot opencv-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.1.10") # use YOUR IP address

import time

got.load_models(["line_recognition", "color_recognition"])
got.set_track_recognition_line(0)

got.transform_adaption_control(False)
got.transform_set_chassis_height(6)

def swipe():
    """Swipe away large cubes (unripe fruit)."""
    got.mechanical_joint_control(0, -35, -60, 500)
    time.sleep(1) # wait 1 second
    got.mechanical_joint_control(-40, -25, -40, 500)
    time.sleep(1) # wait 1 second
    got.mechanical_joint_control(40, -25, -40, 500)
    time.sleep(1) # wait 1 second
    
while True:
    color_info = got.get_color_total_info()
    # print(color_info)
    if color_info[1] == "Cube":
        got.mecanum_stop()
        swipe()
    line_info = got.get_single_track_total_info()
    if line_info:
        offset = line_info[0]
        rot = int(offset*0.5)
        # got.mecanum_move_xyz(0, 20, rot)
        if rot < 0:
            turn = 3 # right
        else:
            turn = 2 # left
        got.transform_move_turn(0, 20, turn, abs(rot))

192.168.1.10:50051


In [ ]:
%pip install ultralytics ugot opencv-python

In [ ]:
import cv2
import numpy as np
import time

from IPython.display import clear_output

from ultralytics import YOLO


# Helper: Draw bounding boxes
def draw_detections(frame, results):
    for r in results:
        boxes = r.boxes  # bounding boxes

        for box in boxes:
            # xyxy format: [x1, y1, x2, y2]
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

            # Confidence & label
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            label = r.names[cls_id]

            # Draw rectangle
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Label text
            text = f"{label} {conf:.2f}"
            cv2.putText(frame, text, (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                        (0, 255, 0), 2)
    return frame


In [4]:
# Use pretrained model from YOLO
model = YOLO("yolo11n.pt")

# Train on new roboflow data
model.train(data=r"C:\Users\miche\TTA-Projects\AIMS cubes.v2-add-version.yolov11\data.yaml", epochs=50, imgsz=512)

New https://pypi.org/project/ultralytics/8.4.105 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.24  Python-3.12.4 torch-2.11.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\miche\TTA-Projects\AIMS cubes.v2-add-version.yolov11\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mos

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000024A2C03CE00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047

In [3]:
trained = YOLO(r"C:\Users\miche\TTA-Projects\runs\detect\train\weights\best.pt")

from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.1.53")

while True:
    frame = got.read_camera_data()
    if frame is not None:
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Run YOLO detection
        results = trained(img, verbose=False)

        # Draw output
        output = draw_detections(img, results)

        # Show
        cv2.imshow("YOLO Detection - AIMS", output)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cv2.destroyAllWindows()

192.168.1.53:50051
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data receiv